# Talos 1B — Chinchilla chat training (20B tokens)

**What this trains.** `talos_1b`, the project's primary 1B model: decoder-only,
**1,036,060,672 parameters**, vocab **49,152**, context **2,048**, hidden 2,048,
16 query / 4 KV heads (4:1 GQA), head_dim 128, RMSNorm pre-norm, RoPE, no
biases, tied embeddings, SDPA attention with the plain fallback. Tokenizer:
`assets/tokenizers/talos_1b_49k.json` (vocab 49,152, 48,892 merges); its sha256
is recorded in every checkpoint and every metrics row.

**Budget.** Chinchilla scaling is ~20 tokens per parameter: 20 × 1,036,060,672
≈ 20.7B, so the owner pinned a round **20,000,000,000** tokens. The loop stops
when the token counter reaches it, however many dataset epochs that takes
(the epoch number is displayed).

**Hardware.** The owner's local **NVIDIA GeForce RTX 3060 12GB** — not Colab
T4. The loop is sized for 12GB: bf16 autocast, gradient checkpointing, gradient
accumulation and 8-bit AdamW. At 3060-class throughput a 20B-token budget is a
long run, so the live table reports measured tok/s and a real ETA, and
checkpoints are frequent enough to stop and resume at will.

**Data.** `HuggingFaceH4/ultrachat_200k`, streamed, linearized to
`User: … / Assistant: …` documents and packed into seq-2048 rows.

**Corpus note.** The 220M-token FineWeb-Edu packed corpus is the 1B
**pretraining** corpus (separate work). This notebook is the chat /
continued-pretraining path on a different dataset: nothing here regenerates,
replaces or silently touches that corpus.

**Resume.** Set `RESUME = True` in the config cell and re-run cells 1–4 and the
training cell; the loop loads the newest `step-<N>.pt` in `OUT_DIR` and
continues from its token counter with the optimizer state, RNG state and epoch
restored.

**Where things go.** `OUT_DIR` defaults to
`/content/drive/MyDrive/Talos/Talos_1B/checkpoints` when Google Drive is
mounted, and to `runs/talos-1b-chat` when it is not; set the `TALOS_1B_OUT`
environment variable to override both. On Drive a disconnect costs no
checkpoints. In it: `step-<N>.pt` — the repo's
`talos-training-checkpoint-v1` schema, so `scripts/eval_checkpoint.py` reads it
directly; `metrics.jsonl` — the full local metric history; `run_state.json` —
the accumulated elapsed time for ETA; and `status.json` / `history.json` — the
tracker payloads described below.

**Public tracker.** Every `TRACKER_MIN_INTERVAL_S` (60 s), and only when
something actually changed, `status.json` + `history.json` are uploaded to the
`PrometheanStudio/talos-training-tracker` **dataset** repo — the owner's
existing public tracker, in its existing schema (`Talos 1B`, 1,036,060,672
params, vocab 49,152, sequence length 2,048, 20B token target, live step /
tokens / losses / newest checkpoint / GPU).

**Run order.** Cells 1–7 are setup (a couple of minutes). Cell 8 is the
training loop.


In [ ]:
# =============================================================================
# 1. Setup — dependencies
# =============================================================================
# torch is deliberately NOT installed here: the 3060 box already has a CUDA
# build, and `pip install torch` would replace it with a CPU-only wheel and the
# run would silently fall back to the CPU. Everything below is a pure-Python
# extra layered on top of the existing torch.
INSTALL = True  # flip to False if the environment is already provisioned

if INSTALL:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
         "datasets>=2.20", "huggingface_hub>=0.24", "bitsandbytes>=0.43",
         "tokenizers>=0.20", "numpy"],
        check=True,
    )
print("dependencies ready")

In [ ]:
# =============================================================================
# 2. Repo + environment
# =============================================================================
import os, sys, json, math, time, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Promethean-Studios/talos.git"
REPO_BRANCH = "main"  # pin a commit sha here (e.g. "c3bb69f") for a frozen run

WORKDIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORKDIR / "talos"
if not (REPO_DIR / "configs").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
for path in (REPO_DIR, REPO_DIR / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
    GPU_CAP = ".".join(map(str, torch.cuda.get_device_capability(0)))
else:
    GPU_NAME, GPU_MEM_GIB, GPU_CAP = "CPU", 0.0, "n/a"

print(f"talos repo : {REPO_DIR}")
print(f"torch      : {torch.__version__}")
print(f"numpy      : {np.__version__}")
print(f"device     : {DEVICE}  {GPU_NAME}  {GPU_MEM_GIB:.1f} GiB  sm{GPU_CAP}")

# The notebook's testable helpers (metrics row, ETA math, live HTML table,
# chat->packed-token adapter, HF Dataset sync). Pure Python: no GPU, no network
# at import time — covered by tests/test_training_lib.py.
from talos_training_lib import (
    TALOS_1B_NAME, TALOS_1B_PARAMS, TALOS_1B_TOKENIZER_ASSET,
    TALOS_1B_VOCAB, TOTAL_TOKENS, MetricsSync, TrackerPublisher,
    build_metrics_row, linearize_dialog, pack_token_stream,
    render_display_table,
)

In [ ]:
# =============================================================================
# 3. Config — every knob lives here
# =============================================================================
# --- budget ------------------------------------------------------------------
# Chinchilla: ~20 tokens per parameter. 20 x 1,036,060,672 = 20.72B, and the
# owner set the round 20B below as the run's total token budget.
TOTAL_TOKENS = 20_000_000_000

# --- model / data shape ------------------------------------------------------
SEQ_LEN         = 2048   # talos_1b context (model.config.max_seq_len)
MICRO_BATCH     = 2      # rows per forward pass; drop to 1 on a CUDA OOM
GRAD_ACCUM      = 16     # one optimizer step per GRAD_ACCUM micro-batches
TOKENS_PER_STEP = MICRO_BATCH * GRAD_ACCUM * SEQ_LEN        # 65,536
TOTAL_STEPS     = TOTAL_TOKENS // TOKENS_PER_STEP           # 305,175

# --- optimisation ------------------------------------------------------------
LR            = 3e-4
MIN_LR_RATIO  = 0.1        # cosine decays to LR * MIN_LR_RATIO
WARMUP_STEPS  = 2_000
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.1
BETAS         = (0.9, 0.95)
USE_8BIT_ADAM = True       # bitsandbytes AdamW8bit; plain AdamW if unavailable

# --- schedule ----------------------------------------------------------------
LOG_EVERY        = 20      # live-table update + metrics row
EVAL_EVERY       = 500     # validation pass (optimizer steps)
EVAL_BATCHES     = 32      # validation micro-batches per eval
CHECKPOINT_EVERY = 1_000   # step-<N>.pt

# --- where things go ---------------------------------------------------------
# Checkpoints on Google Drive survive a disconnect or a runtime reset; with no
# Drive mount, fall back to the repo's runs/ dir. TALOS_1B_OUT overrides both.
DRIVE_ROOT     = Path("/content/drive/MyDrive")
DRIVE_CKPT_DIR = DRIVE_ROOT / "Talos" / "Talos_1B" / "checkpoints"
OUT_DIR = Path(os.environ.get("TALOS_1B_OUT")
               or (DRIVE_CKPT_DIR if DRIVE_ROOT.is_dir() else "runs/talos-1b-chat"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = False             # True -> load the newest step-*.pt and continue

# --- Hugging Face metrics sync ----------------------------------------------
HF_USERNAME        = ""    # blank -> whoami() of the logged-in token
HF_METRICS_REPO    = ""    # blank -> "<whoami>/talos-1b-training-metrics"
SYNC_ENABLED       = True
SYNC_DRY_RUN       = False # True -> build rows locally, never touch the network
SYNC_MIN_INTERVAL_S = 300  # push at most every 5 minutes

# --- public training tracker (the owner's existing dataset repo) --------------
TRACKER_REPO_ID       = "PrometheanStudio/talos-training-tracker"
TRACKER_ENABLED       = True
TRACKER_DRY_RUN       = False  # True -> write status.json/history.json locally only
TRACKER_MIN_INTERVAL_S = 60    # the owner's 100M tracker cadence

# --- dataset -----------------------------------------------------------------
CHAT_DATASET     = "HuggingFaceH4/ultrachat_200k"
CHAT_TRAIN_SPLIT = "train_sft"
CHAT_VAL_SPLIT   = "test_sft"
SHUFFLE_BUFFER   = 10_000
SEED             = 1234

print(json.dumps({
    "total_tokens": TOTAL_TOKENS, "tokens_per_step": TOKENS_PER_STEP,
    "total_steps": TOTAL_STEPS, "seq_len": SEQ_LEN,
    "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
    "lr": LR, "warmup_steps": WARMUP_STEPS,
    "out_dir": str(OUT_DIR), "resume": RESUME,
    "tracker_repo": TRACKER_REPO_ID,
    "chat_dataset": CHAT_DATASET,
}, indent=2))

In [ ]:
# =============================================================================
# 4. Tokenizer + model
# =============================================================================
from configs.presets import ALL_PRESETS
from model.gpt import TalosGPT
from tokenizer.assets import assert_asset_preset_pairing, load_asset
from tokenizer.tokenizer import tokenizer_file_sha256

TOKENIZER_NAME = TALOS_1B_TOKENIZER_ASSET                       # "talos_1b_49k"
TOKENIZER_PATH = REPO_DIR / "assets" / "tokenizers" / f"{TOKENIZER_NAME}.json"

# The asset guard is loud on a mismatched pairing: the 49K asset belongs to
# talos_1b only (the 21l/spec16 variants keep the 32K asset), and a canonical
# 1024-vocab tokenizer must never pair with a 1B model.
assert_asset_preset_pairing(TOKENIZER_NAME, TALOS_1B_NAME)
tokenizer = load_asset(TOKENIZER_NAME)
TOKENIZER_SHA256 = tokenizer_file_sha256(str(TOKENIZER_PATH))
assert tokenizer.vocab_size == TALOS_1B_VOCAB, tokenizer.vocab_size

model_config = ALL_PRESETS[TALOS_1B_NAME]().derive()
model = TalosGPT(model_config)
n_params = model.num_parameters()
# Hard contract, not a print: the preset must build the exact pinned model.
assert n_params == TALOS_1B_PARAMS, (n_params, TALOS_1B_PARAMS)

print(f"tokenizer  : {TOKENIZER_NAME}  vocab={tokenizer.vocab_size:,}  "
      f"merges={tokenizer.num_merges:,}")
print(f"sha256     : {TOKENIZER_SHA256}")
print(f"model      : {TALOS_1B_NAME}  params={n_params:,}  "
      f"vocab={model_config.vocab_size:,}  seq={model_config.max_seq_len:,}")
print(f"layers     : {model_config.num_layers}  hidden={model_config.hidden_size}  "
      f"q/kv heads={model_config.num_attention_heads}/{model_config.num_kv_heads}  "
      f"ffn={model_config.intermediate_size}")

In [ ]:
# =============================================================================
# 5. Data — stream ultrachat_200k, linearize, tokenize, pack to seq-2048 rows
# =============================================================================
from datasets import load_dataset


def packed_rows(split, *, seed, limit_docs=None):
    """One split -> generator of packed `SEQ_LEN` rows of token ids.

    Chat rows are linearized to `User: ...\n\nAssistant: ...` documents, encoded
    with the Talos tokenizer (BOS/EOS per document) and packed end-to-end into
    fixed-length rows by `pack_token_stream`, which bounds-checks every id
    against the 49,152-vocab before it can reach the embedding (repo convention:
    an out-of-range id fails here, with the document in context, never as a
    device-side assert mid-step).
    """
    ds = load_dataset(CHAT_DATASET, split=split, streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=SHUFFLE_BUFFER)
    docs = (linearize_dialog(example["messages"]) for example in ds)
    if limit_docs is not None:
        docs = (doc for i, doc in enumerate(docs) if i < limit_docs)
    ids = (tokenizer.encode(doc, bos=True, eos=True) for doc in docs)
    return pack_token_stream(ids, seq_len=SEQ_LEN, vocab_size=TALOS_1B_VOCAB)


def epoch_batches(split, *, batch_size):
    """Yield (epoch, (batch_size, SEQ_LEN) long tensor) forever.

    Each pass re-shuffles with a different seed, so epoch 2 is not epoch 1 in a
    different order — 20B tokens needs ~100 passes over a ~1B-token corpus.
    """
    epoch = 0
    while True:
        epoch += 1
        buffer = []
        for row in packed_rows(split, seed=SEED + epoch):
            buffer.append(row)
            if len(buffer) == batch_size:
                yield epoch, torch.tensor(buffer, dtype=torch.long)
                buffer = []


def val_batches():
    """Finite validation stream: EVAL_BATCHES micro-batches of held-out chat."""
    buffer = []
    limit = EVAL_BATCHES * MICRO_BATCH * 8      # docs are shorter than 8 rows
    for row in packed_rows(CHAT_VAL_SPLIT, seed=SEED + 999_983, limit_docs=limit):
        buffer.append(row)
        if len(buffer) == MICRO_BATCH:
            yield torch.tensor(buffer, dtype=torch.long)
            buffer = []
    if buffer:
        yield torch.tensor(buffer, dtype=torch.long)


# --- proof the format is right: decode the first packed row ------------------
sample = next(iter(packed_rows(CHAT_TRAIN_SPLIT, seed=SEED)))
print(f"first packed row: {len(sample)} tokens, "
      f"id range [{min(sample)}, {max(sample)}] < vocab {TALOS_1B_VOCAB}")
print("-" * 78)
print(tokenizer.decode(sample[:160]))
print("-" * 78)

In [ ]:
# =============================================================================
# 6. HF Dataset metrics sync
# =============================================================================
# Auth, in order of preference:
#   1. HF_TOKEN in the environment — on Colab: Secrets (key icon) -> add HF_TOKEN,
#      then `from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")`
#      and grant this notebook access to the secret.
#   2. `from huggingface_hub import notebook_login; notebook_login()` (Colab),
#      or `huggingface-cli login` once on the local box.
# The token needs *write* access for the dataset repo to be created/pushed.
from huggingface_hub import whoami

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    HF_USERNAME = HF_USERNAME or whoami(token=HF_TOKEN)["name"]
except Exception as exc:                                   # not logged in
    HF_USERNAME = HF_USERNAME or "unknown"
    print(f"HF Hub not authenticated ({exc}) — metrics stay local until it is")

HF_METRICS_REPO = HF_METRICS_REPO or f"{HF_USERNAME}/talos-1b-training-metrics"

# MetricsSync appends every row to OUT_DIR/metrics.jsonl and pushes that file to
# the dataset repo, at most once per SYNC_MIN_INTERVAL_S. A push failure (bad
# token, no network, repo permissions) is logged and recorded on
# `metrics_sync.last_error`; it never raises into the training loop.
metrics_sync = MetricsSync(
    HF_METRICS_REPO,
    token=HF_TOKEN,
    dry_run=(SYNC_DRY_RUN or not SYNC_ENABLED),
    min_interval_s=SYNC_MIN_INTERVAL_S,
    local_path=str(OUT_DIR / "metrics.jsonl"),
)
print(f"metrics repo : {HF_METRICS_REPO}"
      f"   {'(dry run — no network)' if metrics_sync.dry_run else '(live)'}")
print(f"metrics file : {metrics_sync.local_path}")

# The public tracker: status.json + history.json in the owner's existing schema,
# uploaded to the same dataset repo their 100M run feeds, at most once per
# TRACKER_MIN_INTERVAL_S and only when the payload changed. A tracker failure is
# recorded on `tracker.last_error`; like the metrics sync it never raises into
# the training loop.
tracker = TrackerPublisher(
    TRACKER_REPO_ID,
    token=HF_TOKEN,
    dry_run=(TRACKER_DRY_RUN or not TRACKER_ENABLED),
    min_interval_s=TRACKER_MIN_INTERVAL_S,
    local_dir=str(OUT_DIR),
    tokens_per_step=TOKENS_PER_STEP,
    gpu=GPU_NAME,
)
print(f"tracker repo : {TRACKER_REPO_ID}"
      f"   {'(dry run — no network)' if tracker.dry_run else '(live)'}")
print(f"tracker files: {tracker.status_path} / {tracker.history_path}")


In [ ]:
# =============================================================================
# 7. Optimizer, 12GB memory plan, eval + checkpoint helpers
# =============================================================================
import torch.nn.functional as F
from scripts.train_oasst1 import (
    capture_rng_state, list_checkpoint_candidates, load_checkpoint,
    restore_rng_state, save_checkpoint, validate_resume_checkpoint,
)

# --- 12 GB budget arithmetic (bf16 params, 8-bit AdamW) ----------------------
#   params  (bf16)          1.036B x 2 B  =  2.07 GB
#   grads   (bf16)          1.036B x 2 B  =  2.07 GB
#   AdamW 8-bit moments     1.036B x 2 B  =  2.07 GB
#   -------------------------------------  static  ~6.2 GB
#   activations with gradient checkpointing, MICRO_BATCH 2 x SEQ_LEN 2048
#   (4,096 tokens per micro-batch):
#     saved layer inputs    16 x 2048 x 4096 x 2 B      ~0.27 GB
#     recompute working set ~2-3x that per layer        ~0.6 GB
#     logits + cross-entropy 4096 x 49,152 x (2+4) B    ~1.2 GB
#   -------------------------------------  peak    ~8-9.5 GB of 12 GB
# Gradient checkpointing is what makes it fit: without it the 16 layers of
# stored activations (~4.3 GB at this micro-batch) push the peak past 12 GB.
# On a CUDA OOM set MICRO_BATCH = 1 (~1.5 GB less) and double GRAD_ACCUM, so
# TOKENS_PER_STEP and the whole schedule stay unchanged.
model.set_gradient_checkpointing(True)
model.to(device=DEVICE, dtype=torch.bfloat16)
print(f"grad checkpointing on; model on {DEVICE} as bf16")
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    print(f"allocated after build: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


def build_optimizer():
    """8-bit AdamW on the 3060; plain AdamW is the functional fallback."""
    if USE_8BIT_ADAM:
        try:
            import bitsandbytes as bnb
            return bnb.optim.AdamW8bit(model.parameters(), lr=LR, betas=BETAS,
                                       weight_decay=WEIGHT_DECAY)
        except Exception as exc:
            print(f"bitsandbytes unavailable ({exc}) — using plain AdamW")
    return torch.optim.AdamW(model.parameters(), lr=LR, betas=BETAS,
                             weight_decay=WEIGHT_DECAY)


def lr_at(step):
    """Linear warmup, then cosine decay to LR * MIN_LR_RATIO."""
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    span = max(1, TOTAL_STEPS - WARMUP_STEPS)
    progress = min(1.0, (step - WARMUP_STEPS) / span)
    return LR * (MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * progress)))


@torch.no_grad()
def evaluate():
    """Mean validation loss over EVAL_BATCHES micro-batches of held-out chat."""
    model.eval()
    total, count = 0.0, 0
    for batch in val_batches():
        if count >= EVAL_BATCHES:
            break
        batch = batch.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP_ENABLED):
            logits, _ = model(batch[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(),
                                   batch[:, 1:].reshape(-1))
        total += float(loss)
        count += 1
    model.train()
    return total / count if count else None


def save_step(step, train_loss, val_loss, tokens_consumed, epoch):
    """Write OUT_DIR/step-<N>.pt in the repo's talos-training-checkpoint-v1 schema.

    Weights + optimizer state + RNG state + losses + tokenizer sha256 +
    token budget counters, so `scripts/eval_checkpoint.py` reads it unchanged
    and a resume continues bit-exactly.
    """
    path = OUT_DIR / f"step-{step}.pt"
    save_checkpoint(
        str(path), model, step, train_loss, val_loss, str(TOKENIZER_PATH),
        optimizer=optimizer, rng_state=capture_rng_state(), epoch=epoch,
        tokens_consumed=tokens_consumed, tokenizer_fingerprint=TOKENIZER_SHA256,
    )
    return path.name


print("helpers ready: build_optimizer / lr_at / evaluate / save_step")

In [ ]:
# =============================================================================
# 8. Training loop — 20B tokens, live table, checkpoints, metrics sync
# =============================================================================
# bf16 needs no GradScaler (unlike fp16): the exponent range covers the loss
# scale, so there is nothing to unscale before clipping.
from IPython.display import HTML, display

optimizer = build_optimizer()
# Elapsed time accumulates across sessions (a resumed run's ETA/tok-s must
# reflect the whole run, not just this session).
STATE_PATH = OUT_DIR / "run_state.json"
prior_elapsed = 0.0
if STATE_PATH.exists():
    prior_elapsed = float(json.loads(STATE_PATH.read_text())["elapsed_s"])

step, tokens_consumed, epoch = 0, 0, 0
last_ckpt_name = "—"

if RESUME:
    candidates = list_checkpoint_candidates(str(OUT_DIR))
    if not candidates:
        raise FileNotFoundError(f"RESUME=True but no step-*.pt under {OUT_DIR}")
    ckpt_path = candidates[0]                      # numeric-newest, not lexicographic
    ckpt = load_checkpoint(ckpt_path)
    validate_resume_checkpoint(ckpt, TALOS_1B_NAME)
    model.load_state_dict(ckpt["model_state_dict"])
    if ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt.get("rng_state"):
        restore_rng_state(ckpt["rng_state"])
    step = int(ckpt["step"])
    tokens_consumed = int(ckpt.get("tokens_consumed") or 0)
    epoch = int(ckpt.get("epoch") or 0)
    last_ckpt_name = os.path.basename(ckpt_path)
    tracker.note_checkpoint(step, tokens_consumed)   # history.json starts from here
    print(f"resumed {last_ckpt_name}: step {step}, {tokens_consumed:,} tokens, epoch {epoch}")

display_handle = display(HTML("<em>Talos 1B — waiting for the first log interval…</em>"),
                         display_id="talos-1b-live")
started = time.time()
train_loss = float("nan")
val_loss = None
batches = epoch_batches(CHAT_TRAIN_SPLIT, batch_size=MICRO_BATCH)
model.train()

try:
    while tokens_consumed < TOTAL_TOKENS:
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            epoch, batch = next(batches)
            batch = batch.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16,
                                enabled=AMP_ENABLED):
                logits, _ = model(batch[:, :-1])
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(),
                                       batch[:, 1:].reshape(-1))
            (loss / GRAD_ACCUM).backward()
            step_loss += float(loss)
            tokens_consumed += batch.numel()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        current_lr = lr_at(step)
        for group in optimizer.param_groups:
            group["lr"] = current_lr
        optimizer.step()
        train_loss = step_loss / GRAD_ACCUM

        if step > 0 and step % EVAL_EVERY == 0:
            val_loss = evaluate()

        elapsed_total = prior_elapsed + (time.time() - started)

        if step > 0 and step % CHECKPOINT_EVERY == 0:
            last_ckpt_name = save_step(step, train_loss, val_loss,
                                       tokens_consumed, epoch)
            STATE_PATH.write_text(json.dumps({"elapsed_s": elapsed_total,
                                              "tokens_consumed": tokens_consumed,
                                              "step": step}))
            tracker.note_checkpoint(step, tokens_consumed)

        if step > 0 and step % LOG_EVERY == 0:
            row = build_metrics_row(
                step=step, epoch=max(epoch, 1), tokens_so_far=tokens_consumed,
                total_tokens=TOTAL_TOKENS, elapsed_s=elapsed_total,
                train_loss=train_loss, val_loss=val_loss,
                checkpoint=last_ckpt_name, gpu=GPU_NAME, model=TALOS_1B_NAME,
                params=n_params, vocab=tokenizer.vocab_size, seq_len=SEQ_LEN,
            )
            display_handle.update(HTML(render_display_table(row)))
            # Throttled push; a failure is logged on metrics_sync and swallowed.
            metrics_sync.maybe_push(row)
            # Public tracker: status.json + history.json, same throttled cadence,
            # only-on-change, failures swallowed onto tracker.last_error.
            tracker.maybe_push(row, history_rows=metrics_sync.rows)

        step += 1
except KeyboardInterrupt:
    print("\ninterrupted — writing a checkpoint before stopping")
finally:
    elapsed_total = prior_elapsed + (time.time() - started)
    last_ckpt_name = save_step(step, train_loss, val_loss, tokens_consumed, epoch)
    STATE_PATH.write_text(json.dumps({"elapsed_s": elapsed_total,
                                      "tokens_consumed": tokens_consumed,
                                      "step": step}))
    tracker.note_checkpoint(step, tokens_consumed)
    metrics_sync.flush()
    tracker.flush()          # final status.json/history.json for this session
    print(f"stopped at step {step:,}: {tokens_consumed:,} / {TOTAL_TOKENS:,} tokens "
          f"({100 * tokens_consumed / TOTAL_TOKENS:.2f}%), checkpoint {last_ckpt_name}, "
          f"{len(metrics_sync.rows)} metric row(s), sync errors: {metrics_sync.last_error}, "
          f"tracker: {tracker.last_error or 'ok'}")

### Resume / verify a checkpoint

The loop above already resumes when `RESUME = True` — it loads the numerically
newest `step-*.pt` (never the lexicographically newest: `step-9.pt` must not
beat `step-100.pt`), restores weights, optimizer state and RNG state, and
continues from the checkpoint's own token counter.

Run the cell below to inspect the newest checkpoint before resuming: it applies
the repo's `validate_resume_checkpoint` contract (format, preset, parameter
count, tensor shapes), so a truncated or half-copied file is caught here rather
than 20 minutes into a resumed run.

In [ ]:
# =============================================================================
# 9. Inspect the newest checkpoint (run this before setting RESUME = True)
# =============================================================================
def ckpt_loss(value):
    """Format a checkpoint loss; '—' when that eval has not run yet."""
    return "—" if value is None else f"{float(value):.4f}"


candidates = list_checkpoint_candidates(str(OUT_DIR))
if not candidates:
    print(f"no step-*.pt under {OUT_DIR} yet")
else:
    newest = candidates[0]
    ckpt = load_checkpoint(newest)
    validate_resume_checkpoint(ckpt, TALOS_1B_NAME)
    print(f"newest checkpoint : {os.path.basename(newest)}")
    print(f"format            : {ckpt['format']}")
    print(f"step / epoch      : {ckpt['step']} / {ckpt.get('epoch')}")
    print(f"tokens consumed   : {int(ckpt.get('tokens_consumed') or 0):,}")
    print(f"train / val loss  : {ckpt_loss(ckpt.get('train_loss'))} / {ckpt_loss(ckpt.get('val_loss'))}")
    print(f"params / vocab    : {ckpt['n_params']:,} / {ckpt['vocab_size']:,}")
    print(f"tokenizer sha256  : {ckpt.get('tokenizer_fingerprint')}")
    resume_ready = (ckpt.get("optimizer_state_dict") is not None
                    and ckpt.get("rng_state") is not None)
    print(f"resume-ready      : {resume_ready}")
    print()
    print("To continue: set RESUME = True in the config cell and re-run cell 8.")

In [ ]:
# =============================================================================
# 10. Generation smoke test
# =============================================================================
from inference.generate import generate

model.eval()
prompt = "User: What is the capital of France?\n\nAssistant:"
prompt_ids = torch.tensor([tokenizer.encode(prompt, bos=True)], dtype=torch.long,
                          device=DEVICE)
with torch.no_grad():
    out = generate(model, prompt_ids, max_new_tokens=48, greedy=True,
                   repetition_penalty=1.2, eos_token_id=tokenizer.eos_id)
print(f"prompt : {prompt!r}")
print(f"answer : {tokenizer.decode(out)!r}")
print()
print(f"{len(out)} token(s) generated on {DEVICE} "
      f"({'early EOS stop' if len(out) < 48 else 'length stop'})")

## Summary

- **Model:** `talos_1b` — 1,036,060,672 params, vocab 49,152, context 2,048,
  16Q/4KV GQA, tied embeddings, bf16 + gradient checkpointing + 8-bit AdamW.
- **Budget:** 20,000,000,000 tokens (Chinchilla ≈ 20 tokens/param), 65,536
  tokens per optimizer step.
- **Data:** `HuggingFaceH4/ultrachat_200k`, streamed and packed to seq-2048;
  epochs loop until the token budget is met.
- **Metrics:** every `LOG_EVERY` steps a row lands in `OUT_DIR/metrics.jsonl`
  and is pushed (throttled to `SYNC_MIN_INTERVAL_S`) to the HF dataset repo.
  Row fields: step, epoch, tokens_so_far, total_tokens, tokens_per_sec,
  elapsed_s, eta_s, train_loss, val_loss, checkpoint, gpu, model, params, vocab,
  seq_len, timestamp_utc.
- **Public tracker:** `status.json` + `history.json` in the owner's existing
  schema are uploaded to `PrometheanStudio/talos-training-tracker` every
  `TRACKER_MIN_INTERVAL_S` (60 s) and only when the payload changed, plus once
  at the end of the run. Same keys as the 100M run; the values are 1B values.
- **Checkpoint location:** `/content/drive/MyDrive/Talos/Talos_1B/checkpoints`
  when Google Drive is mounted (a disconnect loses no progress), else
  `runs/talos-1b-chat`; `TALOS_1B_OUT` overrides both.
- **Checkpoints:** `step-<N>.pt` every `CHECKPOINT_EVERY` steps, in the repo's
  `talos-training-checkpoint-v1` schema — `scripts/eval_checkpoint.py` reads
  them directly:

  ```
  python -m scripts.eval_checkpoint --checkpoint runs/talos-1b-chat/step-1000.pt
  ```

- **Sync failures never stop training:** they are logged, recorded on
  `metrics_sync.last_error`, and retried at the next interval.
- **Corpus control:** the 220M-token FineWeb-Edu packed corpus is the separate
  pretraining corpus; this notebook's chat data is a new, distinct dataset and
  replaces nothing.
